In [21]:
from datasets import load_dataset

## Load Dataset

In [22]:
ds = load_dataset('NIH-CARD/CARDBiomedBench', split='test')
ds

Dataset({
    features: ['uuid', 'template_uuid', 'question', 'answer', 'bio_category', 'reasoning_category'],
    num_rows: 10148
})

In [23]:
ds[0]

{'uuid': 'Q1.892',
 'template_uuid': 'Q1',
 'question': 'Is MME a druggable gene?',
 'answer': 'There are currently 2 approved drugs which target MME, indicating that it is a druggable gene.',
 'bio_category': 'Drug Meta',
 'reasoning_category': 'Multi-Filter; Aggregate'}

## Convert to DataFrame

In [24]:
raw_df = ds.to_pandas()
raw_df.head()

,uuid,template_uuid,question,answer,bio_category,reasoning_category
0,Q1.892,Q1,Is MME a druggable gene?,There are currently 2 approved drugs which tar...,Drug Meta,Multi-Filter; Aggregate
1,Q1.669,Q1,Is IL12A a druggable gene?,"There is 1 drug, Ustekinumab, that targets IL1...",Drug Meta,Multi-Filter; Aggregate
2,Q1.1221,Q1,Is RNPEP a druggable gene?,There are currently no approved drugs targetin...,Drug Meta,Multi-Filter; Aggregate
3,Q1.1037,Q1,Is PARP3 a druggable gene?,There are currently 3 approved drugs which tar...,Drug Meta,Multi-Filter; Aggregate
4,Q1.945,Q1,Is NDUFA6 a druggable gene?,There are currently 2 approved drugs which tar...,Drug Meta,Multi-Filter; Aggregate


In [25]:
print('Bio categories:')
for cat in sorted(raw_df['bio_category'].unique()):
    count = (raw_df['bio_category'] == cat).sum()
    print(f'  {cat}: {count}')

print(f'\nReasoning categories:')
for cat in sorted(raw_df['reasoning_category'].unique()):
    count = (raw_df['reasoning_category'] == cat).sum()
    print(f'  {cat}: {count}')

Bio categories:
  Drug Disease Relations; Drug Gene Relations: 270
  Drug Disease Relations; Drug Meta: 270
  Drug Gene Relations: 810
  Drug Gene Relations; Drug Disease Relations: 270
  Drug Meta: 2430
  Drug Meta; Drug Gene Relations: 270
  Drug Meta; Pharmacology: 270
  Gene Disease Relations: 270
  Genomic Location: 810
  Pharmacology: 818
  Pharmacology; Drug Disease Relations; Drug Meta: 270
  SMR Gene Disease Relations: 270
  SMR SNP Disease Relations; SMR Gene Disease Relations: 1080
  SNP Disease Relations: 1080
  SNP Disease Relations; Gene Disease Relations: 540
  SNP Disease Relations; Genomic Location: 22
  Tissue; SMR SNP Disease Relations: 128
  Tissue; SMR SNP Disease Relations; SMR Gene Disease Relations: 270

Reasoning categories:
  Multi-Filter: 540
  Multi-Filter; Aggregate: 270
  Multi-Filter; Aggregate; Join: 270
  Multi-Filter; Data Retrieval: 540
  Multi-Filter; Data Retrieval; Aggregate: 270
  Multi-Filter; Data Retrieval; Aggregate; Join; Threshold; Sorting; 

## Filter for neurobiology-relevant categories

In [26]:
# Keep categories relevant to neurobiology / biomedical RAG evaluation
RELEVANT_BIO_KEYWORDS = [
    'Gene',
    'Disease',
    'Drug',
    'Variant',
    'Pathway',
]

def is_relevant(bio_cat: str) -> bool:
    return any(kw in bio_cat for kw in RELEVANT_BIO_KEYWORDS)

filtered_df = raw_df[raw_df['bio_category'].apply(is_relevant)].reset_index(drop=True)
print(f'After filtering: {len(filtered_df)} (from {len(raw_df)})')
print(f'\nRemaining bio categories:')
print(filtered_df['bio_category'].value_counts())

After filtering: 8520 (from 10148)

Remaining bio categories:
bio_category
Drug Meta                                                        2430
SMR SNP Disease Relations; SMR Gene Disease Relations            1080
SNP Disease Relations                                            1080
Drug Gene Relations                                               810
SNP Disease Relations; Gene Disease Relations                     540
Pharmacology; Drug Disease Relations; Drug Meta                   270
Drug Meta; Pharmacology                                           270
Drug Gene Relations; Drug Disease Relations                       270
Gene Disease Relations                                            270
Drug Disease Relations; Drug Meta                                 270
SMR Gene Disease Relations                                        270
Tissue; SMR SNP Disease Relations; SMR Gene Disease Relations     270
Drug Meta; Drug Gene Relations                                    270
Drug Disease Re

## Transform to question + answer format

In [27]:
df = filtered_df[['question', 'answer', 'bio_category']].copy()
df = df.rename(columns={'bio_category': 'category'})

# Remove duplicates
df = df.drop_duplicates(subset=['question']).reset_index(drop=True)
print(f'After dedup: {len(df)}')

# Remove very short answers
df = df[df['answer'].str.len() >= 200].reset_index(drop=True)
print(f'After filtering short answers: {len(df)}')

df.head(10)

After dedup: 8485
After filtering short answers: 2409


,question,answer,category
0,Are there any NALCN SNPs that have a statistic...,"No, according to the largest European genome-w...",SNP Disease Relations; Gene Disease Relations
1,Are there any OGFOD2 SNPs that have a statisti...,"No, according to genome-wide analysis of Amyot...",SNP Disease Relations; Gene Disease Relations
2,Are there any RP11-63P12.6 SNPs that have a st...,"No, according to the largest European genome-w...",SNP Disease Relations; Gene Disease Relations
3,Are there any HNRNPH3 SNPs that have a statist...,"No, according to genome-wide analysis of Amyot...",SNP Disease Relations; Gene Disease Relations
4,Are there any novel_or_none SNPs that have a s...,"Yes, according the largest European genome-wid...",SNP Disease Relations; Gene Disease Relations
5,Are there any FZD3 SNPs that have a statistica...,"Yes, according to the largest European genome-...",SNP Disease Relations; Gene Disease Relations
6,Are there any FAM105B SNPs that have a statist...,"Yes, according to the largest European genome-...",SNP Disease Relations; Gene Disease Relations
7,Are there any LOC286238 SNPs that have a stati...,"No, according to genome-wide analysis of Amyot...",SNP Disease Relations; Gene Disease Relations
8,Are there any CHAC2 SNPs that have a statistic...,"No, according the largest European genome-wide...",SNP Disease Relations; Gene Disease Relations
9,Are there any RNF40 SNPs that have a statistic...,"Yes, according to the largest European genome-...",SNP Disease Relations; Gene Disease Relations


## Inspect

In [28]:
print(f'Question length — mean: {df["question"].str.len().mean():.0f}, '
      f'median: {df["question"].str.len().median():.0f}')
print(f'Answer length   — mean: {df["answer"].str.len().mean():.0f}, '
      f'median: {df["answer"].str.len().median():.0f}')

for cat in df['category'].unique()[:3]:
    row = df[df['category'] == cat].iloc[0]
    print(f'\n[{cat}]')
    print(f'Q: {row["question"]}')
    print(f'A: {row["answer"]}')

Question length — mean: 99, median: 108
Answer length   — mean: 269, median: 230

[SNP Disease Relations; Gene Disease Relations]
Q: Are there any NALCN SNPs that have a statistically significant association with Alzheimer's Disease?
A: No, according to the largest European genome-wide meta-analysis of Alzheimer's disease (Bellenguez, 2022),  there are no alleles within NALCN that are significantly associated with Alzheimer's Disease.

[SMR SNP Disease Relations; SMR Gene Disease Relations]
Q: Are there any RP11-421M1.8 SNPs that have a statistically significant adjusted SMR p-value in association with Frontotemporal Dementia and what tissues are they significant in?
A: No, based on the Whole Brain, Whole Blood, Frontal Cortex, Cerebellar Hemisphere, Prefrontal Cortex, Cortex, Caudate Basal Ganglia, Tibial Nerve, Hypothalamus, Cerebellum and Nucleus Accumbens Basal samples tested there are no SNPs within the gene RP11-421M1.8 that are significantly associated with Frontotemporal Dement

## Save Data

In [29]:
df.to_csv('cardbiomedbench.csv', index=False)
print(f'Saved {len(df)} rows to cardbiomedbench.csv')

Saved 2409 rows to cardbiomedbench.csv
